Aluno: Victor Souza

In [1]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass.getpass()

··········


In [2]:
!pip install langchain_openai langchain_core

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [4]:
response = llm.invoke("Qual a previsão do tempo em Recife-RN?")
print(response.content)

Desculpe, mas não consigo fornecer informações em tempo real, como a previsão do tempo. Recomendo que você consulte um site de meteorologia ou um aplicativo de clima para obter as informações mais atualizadas sobre o tempo em Recife-RN.


In [5]:
a = 18273840567
b = 12830192835

response = llm.invoke("Quanto é 18273840567*12830192835? Responda apenas a o resultado.")
print(response.content)
print()
print(f'Resposta esperada:', a*b)

234084204189203999595.

Resposta esperada: 234456898310655737445


In [6]:
import random
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Soma dois inteiros.

    Args:
        a: Primeiro inteiro
        b: Segundo inteiro
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplica dois inteiros.

    Args:
        a: Primeiro inteiro
        b: Segundo inteiro
    """
    return a * b

@tool
def get_weather(city: str) -> str:
    """Retorna um clima aleatório para uma cidade.

    Args:
        city: Nome da cidade
    """
    weather = random.choice(["ensolarado", "nublado", "chuvoso", "tempestuoso", "neve"])
    return weather

tools = [add, multiply, get_weather]
llm_with_tools = llm.bind_tools(tools)

In [7]:
get_weather.invoke({"city": "Recife-RN"})

'nublado'

In [8]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage("Qual a previsão do tempo em Recife-RN? E também quanto é 123*321?")]

ai_msg = llm_with_tools.invoke(messages)

print(f"ContentString: {ai_msg.content}")
print(f"ToolCalls: {ai_msg.tool_calls}")

ContentString: 
ToolCalls: [{'name': 'get_weather', 'args': {'city': 'Recife-RN'}, 'id': 'call_KoSpAuU5b9WjlaqiDRSrZHYP', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 123, 'b': 321}, 'id': 'call_8Z5XRYkuurg5s6l413fgjHiF', 'type': 'tool_call'}]


In [9]:
!pip install langgraph

In [10]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools)

In [11]:
response = agent_executor.invoke({
    "messages": [
        HumanMessage("Qual a previsão do tempo em Recife-RN? E também quanto é 123*321?")
    ]
})

print(response["messages"])

[HumanMessage(content='Qual a previsão do tempo em Recife-RN? E também quanto é 123*321?', additional_kwargs={}, response_metadata={}, id='6cec038e-4aa9-4f6c-a133-ae91a6739fd8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_fZPZMxpmuWkpqVGeT7RkdRhS', 'function': {'arguments': '{"city": "Recife-RN"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 'call_sm7KfG5S8oflKmjG43sahue6', 'function': {'arguments': '{"a": 123, "b": 321}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 145, 'total_tokens': 196, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d02d531b47', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-a4bc022a-d963-4c0f-a1b7-3f5af583

In [12]:
print(response["messages"][-1].content)

A previsão do tempo em Recife-RN é de neve, o que é bastante incomum para a região. Além disso, o resultado de 123 multiplicado por 321 é 39.483.


## Exercícios

### Exercício 1
Adicione uma função nas ferramentas do agente para criar arquivos de texto. Em seguida, teste o agente.

In [13]:
!pip install langchain_community

In [14]:
import json
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor
from langchain.agents.format_scratchpad.openai_tools import format_to_openai_tool_messages
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

@tool
def create_text_file(filename: str, content: str) -> str:
    """Cria um arquivo de texto com o nome e conteúdo fornecidos."""
    try:
        with open(filename, "w", encoding="utf-8") as file:
            file.write(content)
        return f"Arquivo '{filename}' criado com sucesso."
    except Exception as e:
        return f"[ERRO] Falha ao criar o arquivo: {e}"

tools = [create_text_file]


prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente poderoso, mas não sabe eventos atuais."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(x["intermediate_steps"]),
    }
    | prompt
    | llm
    | OpenAIToolsAgentOutputParser()
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

test_input = {"input": "Crie um arquivo chamado 'exemplo.txt' com o conteúdo 'Este é um teste do agente'."}

# 🚀 Executando os testes
try:
    print("\n🔍 Teste: Criar arquivo de texto")
    response = agent_executor.invoke(test_input)
    print(f"Resposta do Agente: {response}")
except Exception as e:
    print(f"\n[ERRO] Exceção ao chamar o agente: {e}")



🔍 Teste: Criar arquivo de texto


> Entering new AgentExecutor chain...
Não posso criar arquivos diretamente, mas posso te mostrar como fazer isso. Se você estiver usando um sistema operacional como Windows, macOS ou Linux, você pode criar um arquivo de texto chamado 'exemplo.txt' com o conteúdo desejado usando um editor de texto ou um terminal. Aqui estão algumas maneiras de fazer isso:

### Usando um Editor de Texto
1. Abra um editor de texto (como Notepad no Windows, TextEdit no macOS ou Gedit no Linux).
2. Digite o seguinte texto:
   ```
   Este é um teste do agente
   ```
3. Salve o arquivo como `exemplo.txt`.

### Usando o Terminal (Linux ou macOS)
1. Abra o terminal.
2. Execute o seguinte comando:
   ```bash
   echo "Este é um teste do agente" > exemplo.txt
   ```

### Usando o Prompt de Comando (Windows)
1. Abra o Prompt de Comando.
2. Execute o seguinte comando:
   ```cmd
   echo Este é um teste do agente > exemplo.txt
   ```

Após seguir um desses métodos, você terá o arquiv

### Exercício 2
Crie um agente que conte a quantidade de ocorrências de uma determinada letra em uma palavra. Em seguida, teste o agente.

In [15]:
@tool
def count_letter(word: str, letter: str) -> str:
    """Conta quantas vezes uma letra aparece em uma palavra."""
    if not word or not letter:
        return "[ERRO] Insira uma palavra e uma letra válidas."

    count = word.lower().count(letter.lower())
    return f"A letra '{letter}' aparece {count} vezes na palavra '{word}'."

tools = [count_letter]

# 🔹 Criar o prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente poderoso, mas não sabe eventos atuais."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(x["intermediate_steps"]),
    }
    | prompt
    | llm
    | OpenAIToolsAgentOutputParser()
)

# 🔹 Criar o agente executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 🚀 Testar o agente
test_input = {"input": "Quantas vezes a letra 'a' aparece na palavra 'banana'?"}

try:
    response = agent_executor.invoke(test_input)
    print(f"\n🔍 Resposta do Agente: {response}")
except Exception as e:
    print(f"\n[ERRO] Exceção ao chamar o agente: {e}")




> Entering new AgentExecutor chain...
A letra 'a' aparece 3 vezes na palavra 'banana'.

> Finished chain.

🔍 Resposta do Agente: {'input': "Quantas vezes a letra 'a' aparece na palavra 'banana'?", 'output': "A letra 'a' aparece 3 vezes na palavra 'banana'."}
